# DESeq2 Cross-Species Differential Accessibility (Pseudobulk)

**Goal:** Identify species-specific chromatin accessibility differences using donor-level pseudobulk replicates.

**Pipeline:**
1. Load pseudobulk counts from `12_fragment_matrices/` (output of notebook `08`)
2. Merge across species on shared peak IDs
3. Species-aware peak filtering (min samples per species, max count threshold)
4. Global VST + PCA for quality control
5. Per-cell-type DESeq2 with `~ 0 + species` design (all-vs-one contrasts)
6. Per-cell-type DESeq2 with `~ is_human` design (binary Human vs NonHuman)
7. Species-specific heatmaps and human-peak sharing analysis

**Input:** Parquet files from `cross_species_consensus_v3/12_fragment_matrices/{Species}/`
- `pseudobulk_counts.parquet` — peaks × pseudobulk samples
- `pseudobulk_groups.parquet` — sample metadata (cell_type, donor, label)

**Peak IDs:** Immutable names from BED column 4 (e.g. `unified_000001`, `human_peak_000003`).
After the peak_id fix in `src/fragment_matrices.py`, duplicates should no longer occur.

**Environment:** R kernel with DESeq2, arrow, ggplot2, pheatmap

In [1]:
# =============================================================================
# Cell 1: Load packages
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(arrow)       # read parquet files
  library(ggplot2)
  library(pheatmap)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})
message("All packages loaded")

# =============================================================================
# Cell 2: Configuration 
# =============================================================================
BASE <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"

# ⬇️ UPDATED PATH: Pointing to the new fragment matrices folder
QUANT_DIR <- file.path(BASE, "cross_species_consensus_v3/12_fragment_matrices")

# We'll save the DESeq2 outputs to a new folder so we don't overwrite old stuff
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")


All packages loaded



In [2]:
# =============================================================================
# Cell 3 & 4: Load and merge new Pseudobulk data & metadata
# =============================================================================
# NOTE: After the peak_id fix in src/fragment_matrices.py, region_ids in the
# parquet files are immutable peak names (e.g. "unified_000001") instead of
# coordinate strings. Coordinate-based duplicates (from liftover collisions)
# should no longer occur. The dedup block below is kept as a safety net.
# If you still see duplicates, re-run notebook 08 with FORCE_REBUILD = True.

all_counts <- list()
all_meta <- list()

for (sp in SPECIES) {
  sp_dir <- file.path(QUANT_DIR, sp)
  counts_file <- file.path(sp_dir, "pseudobulk_counts.parquet")
  meta_file <- file.path(sp_dir, "pseudobulk_groups.parquet") 
  
  if (!file.exists(counts_file)) {
    message("Skipping ", sp, ": files not found.")
    next
  }
  
  # Load counts
  counts <- as.data.frame(read_parquet(counts_file))
  
  # Safety net: deduplicate region_ids if any remain
  if (any(duplicated(counts$region_id))) {
    dup_ids <- unique(counts$region_id[duplicated(counts$region_id)])
    message("\n  ⚠️ Found ", length(dup_ids), " duplicated region IDs in ", sp, "!")
    
    # 1. INSPECT: Isolate all rows that share these duplicated IDs
    duplicate_df <- counts[counts$region_id %in% dup_ids, ]
    duplicate_df <- duplicate_df[order(duplicate_df$region_id), ]
    
    message("  🔍 Inspecting the first few columns of the duplicates:")
    print(duplicate_df[, 1:min(4, ncol(duplicate_df))])
    
    # Optional: Save them to a CSV so you can look at the full data
    inspect_file <- file.path(sp_dir, paste0("duplicated_regions_inspect.csv"))
    write.csv(duplicate_df, inspect_file, row.names = FALSE)
    message("  💾 Saved full duplicate dataframe to: ", inspect_file)
    
    # 2. FAST MERGE: Use base R's C-optimized rowsum instead of dplyr
    message("  ⚡ Fast-merging duplicates...")
    numeric_mat <- as.matrix(counts[, -which(names(counts) == "region_id")])
    merged_mat <- rowsum(numeric_mat, group = counts$region_id)
    
    # Rebuild the dataframe
    counts <- as.data.frame(merged_mat)
    counts$region_id <- rownames(counts)
  } else {
    rownames(counts) <- counts$region_id
  }
  
  counts$region_id <- NULL
  
  # Load metadata 
  meta <- as.data.frame(read_parquet(meta_file))
  
  # Prefix the sample labels with the species
  new_labels <- paste0(sp, "_", meta$label)
  colnames(counts) <- new_labels
  
  meta$sample_id <- new_labels
  meta$species <- sp
  
  all_counts[[sp]] <- counts
  all_meta[[sp]] <- meta
  message("Loaded ", sp, ": ", ncol(counts), " pseudobulk samples")
}

# 1. Inner join counts on shared peaks
shared_peaks <- Reduce(intersect, lapply(all_counts, rownames))
counts_merged <- do.call(cbind, unname(lapply(all_counts, function(x) x[shared_peaks, , drop = FALSE])))
counts_merged[is.na(counts_merged)] <- 0

# 2. Bind metadata 
meta_merged <- do.call(rbind, unname(all_meta))
rownames(meta_merged) <- meta_merged$sample_id

# 3. Ensure columns and rows align perfectly
counts_merged <- counts_merged[, rownames(meta_merged)]

message("\nMerged matrix: ", nrow(counts_merged), " peaks x ", ncol(counts_merged), " samples (donor pseudobulks)")

Loaded Human: 254 pseudobulk samples

Loaded Bonobo: 102 pseudobulk samples

Loaded Chimpanzee: 103 pseudobulk samples

Loaded Gorilla: 161 pseudobulk samples

Loaded Macaque: 82 pseudobulk samples

Loaded Marmoset: 112 pseudobulk samples


Merged matrix: 829088 peaks x 814 samples (donor pseudobulks)



In [3]:
head(meta_merged)

,cell_type,donor,region,age,label,n_cells,sample_id,species
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
Human_Adipocytes__B010__Colon__Adult,Adipocytes,B010,Colon,Adult,Adipocytes__B010__Colon__Adult,21,Human_Adipocytes__B010__Colon__Adult,Human
Human_Adipocytes__B012__Colon__Adult,Adipocytes,B012,Colon,Adult,Adipocytes__B012__Colon__Adult,16,Human_Adipocytes__B012__Colon__Adult,Human
Human_BEST4+_cells__B006__Colon__Adult,BEST4+ cells,B006,Colon,Adult,BEST4+_cells__B006__Colon__Adult,116,Human_BEST4+_cells__B006__Colon__Adult,Human
Human_BEST4+_cells__B006__Duodenum__Adult,BEST4+ cells,B006,Duodenum,Adult,BEST4+_cells__B006__Duodenum__Adult,24,Human_BEST4+_cells__B006__Duodenum__Adult,Human
Human_BEST4+_cells__B008__Colon__Adult,BEST4+ cells,B008,Colon,Adult,BEST4+_cells__B008__Colon__Adult,143,Human_BEST4+_cells__B008__Colon__Adult,Human
Human_BEST4+_cells__B008__Duodenum__Adult,BEST4+ cells,B008,Duodenum,Adult,BEST4+_cells__B008__Duodenum__Adult,28,Human_BEST4+_cells__B008__Duodenum__Adult,Human


In [5]:
# =============================================================================
# Cell 5: Setup Experimental Design Metadata & Filter
# =============================================================================
library(dplyr)

# 1. Filter for Adult samples ONLY
meta_merged <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]
message("Subset to Adult samples: ", ncol(counts_merged), " samples remain.")

# Convert string columns to factors for DESeq2
meta_merged$species   <- factor(meta_merged$species, levels = SPECIES)
meta_merged$cell_type <- factor(meta_merged$cell_type)
meta_merged$donor     <- factor(meta_merged$donor)
meta_merged$age       <- factor(meta_merged$age)
meta_merged$region    <- factor(meta_merged$region)

# 2. Calculate total counts (library size) per sample
meta_merged$total_counts <- colSums(counts_merged)

# 3. Summarize total counts by species and cell type
message("\n--- Summary of Total Counts per Species and Cell Type ---")
summary_df <- meta_merged %>%
  group_by(species, cell_type) %>%
  summarize(
    n_samples = n(),
    min_counts = min(total_counts),
    median_counts = median(total_counts),
    max_counts = max(total_counts),
    .groups = "drop"
  )
print(as.data.frame(summary_df))

# 4. Filter samples based on total counts
# IMPORTANT: Adjust this threshold based on the summary printed above!
MIN_SAMPLE_COUNTS <- 500000 
keep_samples <- meta_merged$total_counts >= MIN_SAMPLE_COUNTS

meta_filtered <- meta_merged[keep_samples, ]
counts_filtered <- counts_merged[, rownames(meta_filtered)]

message("\nFiltered out ", sum(!keep_samples), " samples with < ", MIN_SAMPLE_COUNTS, " counts.")

# Setup contrast variable
meta_filtered$is_human <- ifelse(meta_filtered$species == "Human", "Human", "NonHuman")
meta_filtered$is_human <- factor(meta_filtered$is_human, levels = c("NonHuman", "Human"))

# 5. Subset to shared cell types
ct_per_species <- split(as.character(meta_filtered$cell_type), meta_filtered$species)
shared_ct <- Reduce(intersect, ct_per_species)

keep_shared <- meta_filtered$cell_type %in% shared_ct
counts_shared <- counts_filtered[, keep_shared]
meta_shared   <- meta_filtered[keep_shared, ]
meta_shared$cell_type <- droplevels(meta_shared$cell_type)
meta_shared$cell_type <- as.factor(make.names(as.character(meta_shared$cell_type)))

message("Final subset to shared cell types: ", ncol(counts_shared), " pseudobulk samples")
message("Cell types remaining: ", paste(levels(meta_shared$cell_type), collapse = ", "))

Subset to Adult samples: 814 samples remain.


--- Summary of Total Counts per Species and Cell Type ---



       species                           cell_type n_samples min_counts
1        Human                          Adipocytes         2     127854
2        Human                        BEST4+ cells        10      39219
3        Human                         Colonocytes         6     727066
4        Human            Crypt Fibroblasts WNT2B+         9     149526
5        Human                                 ECs        10      52949
6        Human                                EECs        11      66911
7        Human                        Enteric glia         9      19546
8        Human                     Enteric neurons         3      73797
9        Human                         Enterocytes         5      69858
10       Human                        Goblet cells        11     211066
11       Human                                ICCs         6      66741
12       Human                                ILCs         6      15732
13       Human                       Lymphatic ECs        10    


Filtered out 3 samples with < 10000 counts.

Final subset to shared cell types: 675 pseudobulk samples

Cell types remaining: BEST4+ cells, Colonocytes, Crypt Fibroblasts WNT2B+, ECs, EECs, Enteric glia, Enterocytes, Goblet cells, ICCs, Lymphatic ECs, Macrophages, Myofibroblasts, Naive B cells, Paneth cells, Pericytes, Plasma B cells, Specialized Fibroblasts KCNN3+, Specialized Fibroblasts RSPO2/3+, Specialized Fibroblasts RSPO3+ only, Specialized Fibroblasts SYNM+, Stem cells, T cells, TA cells, Tuft cells, Villus Fibroblasts WNT5B+



In [6]:
# =============================================================================
# Cell 6: Global VST and PCA (Quality Control)
# =============================================================================
message("\nPreparing global data for PCA...")

# 6. Filter regions (genes/peaks) with too few counts
# Require at least 10 counts in at least 'min_reps' samples. 
# Adjust 'min_reps' to match your smallest biological replicate group (e.g., 3).
min_reps <- 3 
keep_global_regions <- rowSums(counts_shared >= 10) >= min_reps

message("Filtering regions: Keeping ", sum(keep_global_regions), " out of ", nrow(counts_shared), " regions.")

dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared[keep_global_regions, ]),
  colData   = meta_shared,
  design    = ~ cell_type + is_human
)

# THE FIX: type = "poscounts" prevents the geometric mean error by ignoring zeros
dds_global <- estimateSizeFactors(dds_global, type = "poscounts")
vsd_global <- vst(dds_global, blind = FALSE)

message("VST normalization complete!")


Preparing global data for PCA...

Filtering regions: Keeping 775919 out of 829088 regions.

converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

-- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

VST normalization complete!



In [8]:
# 1. Plot Global PCA
pca_data <- plotPCA(vsd_global, intgroup = c("cell_type", "is_human"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

pca_plot <- ggplot(pca_data, aes(x = PC1, y = PC2, color = cell_type, shape = is_human)) +
  geom_point(size = 3, alpha = 0.8) +
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw() +
  ggtitle("Global PCA of Pseudobulks")

# Save PCA
ggsave(file.path(OUT_DIR, "plots", "Global_PCA.pdf"), pca_plot, width = 8, height = 6)
message("✅ Global PCA saved to plots directory.")



using ntop=500 top features by variance

✅ Global PCA saved to plots directory.



In [12]:
# =============================================================================
# Cell 5: Setup Experimental Design Metadata & Filter
# =============================================================================
library(dplyr)
library(ggplot2)

# 0. CONFIGURATION FOR FILTERING AND AGGREGATION
AGGREGATE_REGIONS <- TRUE       # Set to TRUE to merge (sum) regions per donor/cell_type
MIN_SAMPLE_COUNTS <- 500000     # Threshold for keeping samples

# 1. Filter for Adult samples ONLY
meta_merged <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]
message("Subset to Adult samples: ", ncol(counts_merged), " samples remain.")

# 2. OPTIONAL: Aggregate Regions (Sum counts for the same Donor + Cell Type + Species)
if (AGGREGATE_REGIONS) {
  message("\n--- Aggregating Regions ---")
  # Create a unique ID that ignores the 'region' column
  meta_merged$agg_id <- paste(meta_merged$species, meta_merged$donor, as.character(meta_merged$cell_type), sep = "_")
  
  # Fast rowsum on transposed matrix to sum counts for identical agg_ids
  counts_t <- t(counts_merged)
  merged_t <- rowsum(counts_t, group = meta_merged$agg_id)
  counts_merged <- t(merged_t)
  
  # Rebuild metadata by taking the first instance of each merged group
  meta_merged <- meta_merged %>%
    group_by(agg_id) %>%
    slice(1) %>%
    ungroup() %>%
    mutate(
      region = "Merged", 
      sample_id = agg_id,
      label = agg_id # Update label to match new ID
    ) %>%
    as.data.frame()
  
  rownames(meta_merged) <- meta_merged$sample_id
  # Ensure strict alignment
  counts_merged <- counts_merged[, rownames(meta_merged)]
  message("Aggregated down to ", ncol(counts_merged), " unique donor/cell-type pseudobulks.")
}

# 3. Calculate total counts (library size) per sample
meta_merged$total_counts <- colSums(counts_merged)

# 4. Print Top and Bottom 3 samples per Species
message("\n--- Top & Bottom 3 Samples by Total Counts per Species ---")
top_bottom_df <- meta_merged %>%
  group_by(species) %>%
  arrange(desc(total_counts)) %>%
  slice(c(1:3, (n()-2):n())) %>%
  mutate(Rank = c("Top 1", "Top 2", "Top 3", "Bottom 3", "Bottom 2", "Bottom 1")) %>%
  select(species, Rank, sample_id, total_counts, cell_type) %>%
  ungroup()

print(as.data.frame(top_bottom_df))

# 5. Plot Total Counts Histogram with Threshold Line (Log-transformed)
plots_dir <- file.path(OUT_DIR, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

hist_plot <- ggplot(meta_merged, aes(x = total_counts, fill = species)) +
  geom_histogram(bins = 30, color = "black", alpha = 0.8) +
  # ggplot2 automatically maps the vline to the log scale when we use scale_x_log10
  geom_vline(xintercept = MIN_SAMPLE_COUNTS, color = "red", linetype = "dashed", linewidth = 1) +
  facet_wrap(~ species, scales = "free_y") +
  theme_bw() +
  scale_x_log10(labels = scales::comma) +  # <-- The magic log-transform line
  labs(
    title = "Distribution of Total Counts per Pseudobulk Sample",
    subtitle = paste("Red dashed line = threshold (", MIN_SAMPLE_COUNTS, ")"),
    x = "Total Counts (log10 scale)",
    y = "Number of Samples"
  ) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1) # Angles labels if they get crowded
  )

ggsave(file.path(plots_dir, "Count_Distributions_Per_Species_Log10.pdf"), hist_plot, width = 10, height = 6)
message("✅ Log-transformed count distribution plot saved to: ", file.path(plots_dir, "Count_Distributions_Per_Species_Log10.pdf"))

# 6. Apply filtering
keep_samples <- meta_merged$total_counts >= MIN_SAMPLE_COUNTS
meta_filtered <- meta_merged[keep_samples, ]
counts_filtered <- counts_merged[, rownames(meta_filtered)]
message("\nFiltered out ", sum(!keep_samples), " samples with < ", MIN_SAMPLE_COUNTS, " counts.")

# 7. Setup factors and shared cell types
meta_filtered$species   <- factor(meta_filtered$species, levels = SPECIES)
meta_filtered$donor     <- factor(meta_filtered$donor)
meta_filtered$age       <- factor(meta_filtered$age)
meta_filtered$region    <- factor(meta_filtered$region)
meta_filtered$is_human  <- factor(ifelse(meta_filtered$species == "Human", "Human", "NonHuman"), levels = c("NonHuman", "Human"))

# Clean up cell type names for DESeq2 safety
meta_filtered$cell_type <- as.factor(make.names(as.character(meta_filtered$cell_type)))

ct_per_species <- split(as.character(meta_filtered$cell_type), meta_filtered$species)
shared_ct <- Reduce(intersect, ct_per_species)

keep_shared <- meta_filtered$cell_type %in% shared_ct
counts_shared <- counts_filtered[, keep_shared]
meta_shared   <- meta_filtered[keep_shared, ]
meta_shared$cell_type <- droplevels(meta_shared$cell_type)

message("Final subset to shared cell types: ", ncol(counts_shared), " pseudobulk samples")

Subset to Adult samples: 814 samples remain.


--- Aggregating Regions ---

Aggregated down to 674 unique donor/cell-type pseudobulks.


--- Top & Bottom 3 Samples by Total Counts per Species ---



      species     Rank                                     sample_id
1       Human    Top 1                           Human_B010_TA cells
2       Human    Top 2                           Human_B012_TA cells
3       Human    Top 3                           Human_B008_TA cells
4       Human Bottom 3                          Human_B011_Monocytes
5       Human Bottom 2                               Human_B011_ILCs
6       Human Bottom 1                     Human_B006_Plasma B cells
7      Bonobo    Top 1                    Bonobo_BN114_Naive B cells
8      Bonobo    Top 2                          Bonobo_BN131_T cells
9      Bonobo    Top 3                         Bonobo_BN114_TA cells
10     Bonobo Bottom 3 Bonobo_BN114_Specialized Fibroblasts RSPO2/3+
11     Bonobo Bottom 2                       Bonobo_BN133_Adipocytes
12     Bonobo Bottom 1                     Bonobo_BN116_Enteric glia
13 Chimpanzee    Top 1                      Chimpanzee_BN083_T cells
14 Chimpanzee    Top 2            

✅ Count distribution plot saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Count_Distributions_Per_Species.pdf


Filtered out 334 samples with < 5e+05 counts.

Final subset to shared cell types: 224 pseudobulk samples



In [ ]:
# =============================================================================
# Cell 6: Species-Aware Peak Filtering
# =============================================================================
MIN_SAMPLES_PER_SPECIES <- 2  # How many samples within a species need signal?
MIN_SPECIES_ACTIVE <- 6      # How many species must pass the above threshold?

n_before <- nrow(counts_shared)

# 1. Filter: Max count across any sample > 150 (removes total background noise)
keep_count <- apply(counts_shared, 1, max) >= 150

# 2. Filter: Species-aware presence
active_in_species <- sapply(SPECIES, function(sp) {
  sp_cols <- meta_shared$species == sp
  if (sum(sp_cols) > 0) {
    rowSums(counts_shared[, sp_cols, drop = FALSE] >= 10) >= MIN_SAMPLES_PER_SPECIES
  } else {
    rep(FALSE, nrow(counts_shared))
  }
})

# A peak is kept if it is active in at least MIN_SPECIES_ACTIVE
keep_species_thresh <- rowSums(active_in_species) >= MIN_SPECIES_ACTIVE

keep_peaks <- keep_count & keep_species_thresh
counts_shared <- counts_shared[keep_peaks, ]

message("Species-Aware Peak filtering:")
message("  Kept: ", sum(keep_peaks), " (", round(100 * sum(keep_peaks) / n_before, 1), "%)")

# =============================================================================
# Cell 6b: Global VST and Custom PCA (Post-Filtering)
# =============================================================================
message("\nPreparing global data for PCA based on species-aware filtered peaks...")

# Run a global DESeq2 model purely to get the normalized, log-scaled data (VST)
dds_global_filt <- DESeqDataSetFromMatrix(
  countData = round(counts_shared),
  colData   = meta_shared,
  design    = ~ species + cell_type
)

# THE FIX: Apply type="poscounts" here as well!
dds_global_filt <- estimateSizeFactors(dds_global_filt, type = "poscounts")
vsd_global_filt <- vst(dds_global_filt, blind = FALSE)

message("Extracting PCA coordinates...")

# 1. Extract raw PCA data with ALL metadata fields
pca_data <- plotPCA(vsd_global_filt, intgroup = c("cell_type", "species", "donor", "region", "age"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# 2. Filter and collapse cell types for visualization
# NOTE: Updated to match make.names() formatting from Cell 5!
target_cts <- c("Enterocytes", "Colonocytes", "Goblet.cells", "T.cells", "Naive.B.cells", "Plasma.B.cells")

pca_filtered <- pca_data %>%
  mutate(cell_type_broad = case_when(
    cell_type %in% target_cts ~ as.character(cell_type),
    grepl("Fibroblast", cell_type) ~ "Fibroblasts", 
    TRUE ~ "Drop"
  )) %>%
  filter(cell_type_broad != "Drop")

pca_filtered$cell_type_broad <- factor(pca_filtered$cell_type_broad)

message(sprintf("Plotting PCA for %d selected samples...", nrow(pca_filtered)))

# 3. Base Plotting Function for Conserved Peaks PCA
plot_custom_pca_post <- function(df, color_var, shape_var = NULL, title, add_labels = FALSE) {
  p <- ggplot(df, aes_string(x = "PC1", y = "PC2", color = color_var, shape = shape_var)) +
    geom_point(size = 4, alpha = 0.85) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    theme_bw() +
    ggtitle(title) +
    theme(
      legend.position = "right",
      legend.title = element_text(face = "bold"),
      plot.title = element_text(face = "bold", size = 14)
    )
  
  # Only add donor labels to the combined plot to prevent total clutter
  if (add_labels) {
    p <- p + ggrepel::geom_text_repel(aes(label = donor), size = 3, show.legend = FALSE, max.overlaps = 20)
  }
  
  # Handle shapes if provided
  if (!is.null(shape_var)) {
    num_shapes <- length(unique(df[[shape_var]]))
    # Using your preferred shape values + some extras just in case
    p <- p + scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8, 9)[1:num_shapes]) 
  }
  return(p)
}

# 4. Generate and save all requested PCA variations
pca_plots_post <- list(
  Species   = plot_custom_pca_post(pca_filtered, color_var = "species", title = "Conserved Peaks PCA: Species"),
  CellType  = plot_custom_pca_post(pca_filtered, color_var = "cell_type_broad", title = "Conserved Peaks PCA: Cell Type"),
  Region    = plot_custom_pca_post(pca_filtered, color_var = "region

In [13]:
# =============================================================================
# Cell 6: Global VST and Comprehensive PCA
# =============================================================================
message("\nPreparing global data for PCA...")

# 1. Filter regions (require at least 10 counts in at least 3 samples)
min_reps <- 3 
keep_global_regions <- rowSums(counts_shared >= 10) >= min_reps
message("Filtering regions: Keeping ", sum(keep_global_regions), " out of ", nrow(counts_shared), " regions.")

# 2. Build DESeq object & VST
dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared[keep_global_regions, ]),
  colData   = meta_shared,
  design    = ~ species + cell_type
)

# Use type="poscounts" to prevent zero-inflation errors
dds_global <- estimateSizeFactors(dds_global, type = "poscounts")
vsd_global <- vst(dds_global, blind = FALSE)
message("VST normalization complete!")

# 3. Extract PCA coordinates encompassing ALL metadata we care about
pca_data <- plotPCA(vsd_global, intgroup = c("species", "cell_type", "region", "age", "donor"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# Base plotting function to keep the code clean
plot_custom_pca <- function(df, color_var, shape_var = NULL, title) {
  p <- ggplot(df, aes_string(x = "PC1", y = "PC2", color = color_var, shape = shape_var)) +
    geom_point(size = 4, alpha = 0.8) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    theme_bw() +
    ggtitle(title) +
    theme(
      legend.position = "right",
      legend.title = element_text(face = "bold"),
      plot.title = element_text(face = "bold", size = 14)
    )
  
  # If a shape is used and has many levels, allow up to 15 default shapes
  if (!is.null(shape_var)) {
    num_shapes <- length(unique(df[[shape_var]]))
    if (num_shapes > 6) {
      p <- p + scale_shape_manual(values = 1:num_shapes)
    }
  }
  return(p)
}

# 4. Generate and save all requested PCA variations
message("\nGenerating PCA Plots...")

pca_plots <- list(
  Species   = plot_custom_pca(pca_data, color_var = "species", title = "PCA colored by Species"),
  CellType  = plot_custom_pca(pca_data, color_var = "cell_type", title = "PCA colored by Cell Type"),
  Region    = plot_custom_pca(pca_data, color_var = "region", title = "PCA colored by Region"),
  Age       = plot_custom_pca(pca_data, color_var = "age", title = "PCA colored by Age"),
  Donor     = plot_custom_pca(pca_data, color_var = "donor", title = "PCA colored by Donor"),
  Combined  = plot_custom_pca(pca_data, color_var = "species", shape_var = "cell_type", title = "PCA: Species (Color) & Cell Type (Shape)")
)

for (plot_name in names(pca_plots)) {
  file_name <- file.path(plots_dir, paste0("Global_PCA_", plot_name, ".pdf"))
  ggsave(file_name, pca_plots[[plot_name]], width = 10, height = 7)
  message("  Saved: ", file_name)
}
message("✅ All PCA plots generated successfully.")


Preparing global data for PCA...

Filtering regions: Keeping 789572 out of 829088 regions.

converting counts to integer mode

VST normalization complete!

using ntop=500 top features by variance


Generating PCA Plots...

Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.”
  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Global_PCA_Species.pdf

  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Global_PCA_CellType.pdf

  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Global_PCA_Region.pdf

  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2

In [14]:
# =============================================================================
# Cell 6: Species-Aware Peak Filtering
# =============================================================================
MIN_SAMPLES_PER_SPECIES <- 2  # How many samples within a species need signal?
MIN_SPECIES_ACTIVE <- 6      # How many species must pass the above threshold?

n_before <- nrow(counts_shared)

# 1. Filter: Max count across any sample > 150 (removes total background noise)
keep_count <- apply(counts_shared, 1, max) >= 150

# 2. Filter: Species-aware presence
# Create a logical matrix: Peaks (rows) x Species (cols)
active_in_species <- sapply(SPECIES, function(sp) {
  sp_cols <- meta_shared$species == sp
  if (sum(sp_cols) > 0) {
    # TRUE if at least MIN_SAMPLES_PER_SPECIES have >= 10 fragments
    rowSums(counts_shared[, sp_cols, drop = FALSE] >= 10) >= MIN_SAMPLES_PER_SPECIES
  } else {
    rep(FALSE, nrow(counts_shared))
  }
})

# A peak is kept if it is active in at least MIN_SPECIES_ACTIVE
keep_species_thresh <- rowSums(active_in_species) >= MIN_SPECIES_ACTIVE

keep_peaks <- keep_count & keep_species_thresh
counts_shared <- counts_shared[keep_peaks, ]

message("Species-Aware Peak filtering:")
message("  Kept: ", sum(keep_peaks), " (", round(100 * sum(keep_peaks) / n_before, 1), "%)")

# =============================================================================
# Cell 6b: Global VST and Custom PCA (Post-Filtering)
# =============================================================================
message("\nPreparing global data for PCA based on species-aware filtered peaks...")

# Run a global DESeq2 model purely to get the normalized, log-scaled data (VST)
dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared),
  colData   = meta_shared,
  design    = ~ species + cell_type
)

# Estimate size factors and apply Variance Stabilizing Transformation (VST)
dds_global <- estimateSizeFactors(dds_global)
vsd_global <- vst(dds_global, blind = FALSE)

message("Extracting PCA coordinates...")

# 1. Extract raw PCA data 
pca_data <- plotPCA(vsd_global, intgroup = c("cell_type", "species", "donor"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# 2. Filter and collapse cell types for visualization
target_cts <- c("Enterocytes", "Colonocytes", "Goblet cells", "T cells", "Naive B cells", "Plasma B cells")

pca_filtered <- pca_data %>%
  mutate(cell_type_broad = case_when(
    cell_type %in% target_cts ~ as.character(cell_type),
    grepl("Fibroblast", cell_type) ~ "Fibroblasts", # Collapse all fibroblast subtypes
    TRUE ~ "Drop"
  )) %>%
  filter(cell_type_broad != "Drop")

pca_filtered$cell_type_broad <- factor(pca_filtered$cell_type_broad)

message(sprintf("Plotting PCA for %d selected samples...", nrow(pca_filtered)))

# 3. Build the custom plot
# - Color: Species
# - Shape: Cell Type
# - Label: Donor
pca_plot <- ggplot(pca_filtered, aes(x = PC1, y = PC2, color = species, shape = cell_type_broad)) +
  geom_point(size = 4, alpha = 0.85) +
  ggrepel::geom_text_repel(aes(label = donor), size = 3, show.legend = FALSE, max.overlaps = 20) +
  
  # Manually set 7 distinct shapes
  scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8)) + 
  
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw() +
  ggtitle("Global PCA (Conserved Peaks)") +
  theme(
    legend.position = "right",
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold", size = 14)
  )

# Save the plot
ggsave(file.path(OUT_DIR, "plots", "Custom_Filtered_PCA_2.pdf"), pca_plot, width = 10, height = 7)
message("✅ Custom PCA saved to: ", file.path(OUT_DIR, "plots", "Custom_Filtered_PCA_2.pdf"))

Species-Aware Peak filtering:

  Kept: 71617 (8.6%)


Preparing global data for PCA based on species-aware filtered peaks...

converting counts to integer mode

Extracting PCA coordinates...

using ntop=500 top features by variance

Plotting PCA for 78 selected samples...

Warning message:
“ggrepel: 1 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
✅ Custom PCA saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Custom_Filtered_PCA_2.pdf



In [ ]:
# =============================================================================
# Cell 7: Per-Cell-Type DESeq2 (Testing ALL Species)
# =============================================================================
res_list <- list() 

cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  message(sprintf("\n=== Processing %s ===", ct))
  
  meta_ct <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]
  
  # Ensure the cell type is actually present in multiple species
  present_species <- unique(as.character(meta_ct$species))
  if (length(present_species) < 2) {
    message("  ⚠️ Skipping: Not enough species represented for this cell type.")
    next
  }
  
  # Local peak filtering for this specific cell type
  keep_ct <- rowSums(counts_ct >= 10) >= 3
  counts_ct_filtered <- counts_ct[keep_ct, ]
  
  # ⬇️ THE FIX 1: Use ~ 0 + species (removes intercept for clean contrasts)
  dds_ct <- DESeqDataSetFromMatrix(
    countData = round(counts_ct_filtered),
    colData   = meta_ct,
    design    = ~ 0 + species
  )
  
  dds_ct <- DESeq(dds_ct, quiet = TRUE)
  
  res_list[[ct]] <- list()
  
  # Get the exact term names (e.g., "speciesHuman", "speciesMacaque")
  res_names <- resultsNames(dds_ct)
  
  for (target_sp in present_species) {
    
    target_term <- paste0("species", target_sp)
    if (!target_term %in% res_names) next
    
    # Build a numeric contrast vector: Target species = 1, All others = -1 / (N-1)
    contrast_vec <- rep(-1 / (length(res_names) - 1), length(res_names))
    target_idx <- which(res_names == target_term)
    contrast_vec[target_idx] <- 1
    
    # ⬇️ THE FIX 2: Pass the numeric vector directly, not inside a list()
    res_sp <- results(dds_ct, contrast = contrast_vec)
    res_sp_ordered <- res_sp[order(res_sp$padj), ]
    
    # Save it to our nested list 
    res_list[[ct]][[target_sp]] <- res_sp_ordered
    
    sig_hits <- sum(res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, na.rm = TRUE)
    message(sprintf("  ✅ %-12s specific regions: %d (padj < 0.05, LFC > 1)", target_sp, sig_hits))
  }
}

In [ ]:
# =============================================================================
# Cell 8: Top Species-Specific Heatmap (Enterocytes)
# =============================================================================
target_ct <- "Enterocytes"
message("\n=== Generating Species-Specific Heatmap for: ", target_ct, " ===")

top_peaks_list <- list()

# Extract top 25 significant regions per species (padj < 0.05, LFC > 1)
for (sp in names(res_list[[target_ct]])) {
  res_sp <- res_list[[target_ct]][[sp]]
  
  # Filter for significance and positive LFC (means higher in THIS species)
  sig_res <- res_sp[!is.na(res_sp$padj) & res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, ]
  
  # Sort by fold change to get the strongest biological markers
  sig_res <- sig_res[order(sig_res$log2FoldChange, decreasing = TRUE), ] 
  top_peaks_list[[sp]] <- rownames(head(sig_res, 25))
}

# Combine and get unique peaks (max 150 regions if 6 species * 25)
top_regions <- unique(unlist(top_peaks_list))

# Subset metadata and VST data for Enterocytes
meta_ct <- meta_shared[meta_shared$cell_type == target_ct, ]

# Sort metadata by species so the heatmap columns form neat species blocks
meta_ct <- meta_ct[order(meta_ct$species), ]
samples_to_plot <- rownames(meta_ct)

# Get log-normalized values from the global VST object we made earlier
mat <- assay(vsd_global)[top_regions, samples_to_plot]

# Row scaling (Z-score) so high/low peaks are visually comparable
mat_scaled <- t(scale(t(mat)))

# Create Annotation dataframe
anno_col <- data.frame(
  Species = meta_ct$species,
  row.names = samples_to_plot
)

# Draw Heatmap
heatmap_file <- file.path(OUT_DIR, "plots", paste0("Heatmap_", target_ct, "_Species_Markers.pdf"))

pheatmap(mat_scaled,
         annotation_col = anno_col,
         cluster_cols = FALSE,  # 👈 FALSE keeps the species neatly grouped together!
         cluster_rows = TRUE,   # Groups similar peaks together
         show_rownames = FALSE,
         show_colnames = FALSE,
         main = paste("Top Species-Specific Regions in", target_ct),
         filename = heatmap_file,
         width = 9, height = 7)

message("✅ Heatmap saved to: ", heatmap_file)



In [ ]:
# =============================================================================
# Cell 9: Human-Specific Regions - Cell Type Sharing Analysis
# =============================================================================
message("\n=== Analyzing Human-Specific Region Sharing Across Cell Types ===")

target_species <- "Human"
human_hits_per_ct <- list()

# Collect all Human-specific hits from every cell type
for (ct in names(res_list)) {
  if (target_species %in% names(res_list[[ct]])) {
    res <- res_list[[ct]][[target_species]]
    
    # Filter for significant Human hits
    sig_peaks <- rownames(res[!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange > 1, ])
    if (length(sig_peaks) > 0) {
      human_hits_per_ct[[ct]] <- sig_peaks
    }
  }
}

# Master list of all unique human peaks
all_human_peaks <- unique(unlist(human_hits_per_ct))
message("Total unique human-specific peaks across all cell types: ", length(all_human_peaks))

# Create a logical presence/absence matrix (Peaks x CellTypes)
overlap_mat <- sapply(human_hits_per_ct, function(peaks) {
  all_human_peaks %in% peaks
})
rownames(overlap_mat) <- all_human_peaks

# Count how many cell types each peak is active in
peak_ct_counts <- rowSums(overlap_mat)

# Print Summary
sharing_summary <- table(peak_ct_counts)
message("\nDistribution of Human Peak Sharing:")
for (i in seq_along(sharing_summary)) {
  message(sprintf("  Active in exactly %s cell type(s): %d peaks", names(sharing_summary)[i], sharing_summary[i]))
}

# Plot the distribution
sharing_df <- data.frame(CellTypesCount = as.numeric(names(sharing_summary)),
                         NumPeaks = as.numeric(sharing_summary))

bar_plot <- ggplot(sharing_df, aes(x = factor(CellTypesCount), y = NumPeaks)) +
  geom_bar(stat = "identity", fill = "indianred") +
  geom_text(aes(label = NumPeaks), vjust = -0.5, size = 3.5) +
  theme_classic() +
  labs(title = "Human-Specific Regions: Sharing Across Cell Types",
       x = "Number of Cell Types", y = "Number of Regions") +
  theme(plot.title = element_text(face = "bold"))

ggsave(file.path(OUT_DIR, "plots", "Human_Peak_Sharing_Summary.pdf"), bar_plot, width = 7, height = 5)
message("✅ Sharing summary barplot saved to plots directory.")

In [ ]:
# =============================================================================
# Cell 7: Per-Cell-Type DESeq2 Loop
# =============================================================================
# We will store the DESeq2 results for each cell type in this list
res_list <- list()

cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  message(sprintf("\n=== Processing %s ===", ct))
  
  # 1. Subset metadata and counts
  meta_ct <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]
  
  # Ensure we have both Human and NonHuman samples to compare
  if (length(unique(meta_ct$is_human)) < 2) {
    message("  ⚠️ Skipping: Needs both Human and NonHuman samples.")
    next
  }
  
  # 2. Filter peaks SPECIFICALLY for this cell type
  # Keep peaks that have at least 10 counts in at least 3 samples of this cell type
  keep_ct <- rowSums(counts_ct >= 10) >= 3
  counts_ct_filtered <- counts_ct[keep_ct, ]
  
  message("  Testing ", nrow(counts_ct_filtered), " active peaks across ", ncol(counts_ct_filtered), " samples.")
  
  # 3. Run DESeq2
  # NOTE: Using ~ is_human. If you want to control for region here, you can try 
  # ~ region + is_human, but it will fail if a region lacks Human/NonHuman balance.
  dds_ct <- DESeqDataSetFromMatrix(
    countData = round(counts_ct_filtered),
    colData   = meta_ct,
    design    = ~ is_human
  )
  
  dds_ct <- DESeq(dds_ct, quiet = TRUE)
  res_ct <- results(dds_ct, contrast = c("is_human", "Human", "NonHuman"))
  
  # Order by significance and save
  res_ct_ordered <- res_ct[order(res_ct$padj), ]
  res_list[[ct]] <- res_ct_ordered
  
  # Print quick summary
  sig_hits <- sum(res_ct$padj < 0.05 & abs(res_ct$log2FoldChange) > 1, na.rm = TRUE)
  message("  ✅ Found ", sig_hits, " significant human-specific regions (padj < 0.05, |LFC| > 1).")
}

# =============================================================================
# Cell 8: Visualize Top Hits in a Heatmap
# =============================================================================
# Let's pick a specific cell type to visualize (e.g., the first one in the list)
target_ct <- cell_types[1] # You can change this to "Enterocytes", "Stem_cells", etc.
message("\nGenerating heatmap for top regions in: ", target_ct)

# Get the top 50 significant regions for this cell type
res_target <- res_list[[target_ct]]
top_regions <- rownames(head(res_target[which(res_target$padj < 0.05), ], 50))

if (length(top_regions) < 2) {
  message("  ⚠️ Not enough significant regions to draw a heatmap.")
} else {
  # Subset the GLOBAL variance-stabilized data for just these peaks and samples
  samples_to_plot <- rownames(meta_shared[meta_shared$cell_type == target_ct, ])
  mat <- assay(vsd_global)[top_regions, samples_to_plot]
  
  # Scale the data by row (Z-score) so high/low peaks are comparable visually
  mat_scaled <- t(scale(t(mat)))
  
  # Set up the annotation bar at the top of the heatmap
  anno_col <- data.frame(
    Species = meta_shared[samples_to_plot, "species"],
    Region = meta_shared[samples_to_plot, "region"],
    row.names = samples_to_plot
  )
  
  # Plot and save
  heatmap_file <- file.path(OUT_DIR, "plots", paste0("Heatmap_", target_ct, "_Top50.pdf"))
  
  pheatmap(mat_scaled, 
           annotation_col = anno_col,
           cluster_rows = TRUE, 
           cluster_cols = TRUE, 
           show_rownames = FALSE, # Set TRUE if you want to see the unified_ IDs, but 50 gets crowded
           show_colnames = FALSE,
           main = paste("Top 50 Human-Specific Regions in", target_ct),
           filename = heatmap_file,
           width = 8, height = 6)
  
  message("✅ Heatmap saved to: ", heatmap_file)
}